In [71]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIG
# ============================================================

DATA_ROOT = "/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color"

TRAIN_USERS = [2, 4, 5, 6, 7, 8, 9, 16, 17, 18, 19, 20, 21, 22, 24]
VAL_USERS   = [1, 3, 23]

NUM_CLASSES = 40

# Temporal
NUM_FRAMES = 32

# Spatial
IMAGE_SIZE = 160

# DataLoader
BATCH_SIZE = 16
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [72]:
!pip install wandb -q

import os
import random
import numpy as np
import torch

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("DEVICE")
print("=" * 60)

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

    # Better matmul performance on modern NVIDIA GPUs
    torch.set_float32_matmul_precision("high")

else:
    print("WARNING: CUDA is NOT available.")

# ------------------------------------------------------------
# W&B
# ------------------------------------------------------------
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

DEVICE
PyTorch version: 2.10.0+cu128
Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [73]:
jet_bgr = cv2.applyColorMap(
    np.arange(256, dtype=np.uint8).reshape(-1, 1),
    cv2.COLORMAP_JET
).reshape(256, 3)

# OpenCV gives BGR.
# Convert to RGB.
JET_RGB = jet_bgr[:, ::-1]

# Pack RGB into a single integer:
# RRRRRRRR GGGGGGGG BBBBBBBB
JET_KEYS = (
    (JET_RGB[:, 0].astype(np.uint32) << 16) |
    (JET_RGB[:, 1].astype(np.uint32) << 8) |
    JET_RGB[:, 2].astype(np.uint32)
)

# key -> depth index
JET_LUT = np.full(256**3, -1, dtype=np.int16)
JET_LUT[JET_KEYS] = np.arange(256, dtype=np.int16)

print("JET palette:", JET_RGB.shape)
print("LUT:", JET_LUT.shape)
print("Valid JET colors:", np.sum(JET_LUT >= 0))

JET palette: (256, 3)
LUT: (16777216,)
Valid JET colors: 256


In [74]:
def decode_jet_depth(img_rgb):
    """
    img_rgb:
        H x W x 3 uint8 RGB image

    Returns:
        depth: H x W float32 in [0, 1]
        mask : H x W float32
    """

    rgb = img_rgb.astype(np.uint32)

    keys = (
        (rgb[:, :, 0] << 16) |
        (rgb[:, :, 1] << 8) |
        rgb[:, :, 2]
    )

    depth_idx = JET_LUT[keys]

    # Valid depth pixels are exact JET colors.
    mask = (depth_idx >= 0).astype(np.float32)

    # Convert 0..255 -> 0..1
    depth = np.maximum(depth_idx, 0).astype(np.float32) / 255.0

    # Invalid/background pixels -> 0
    depth *= mask

    return depth, mask

In [75]:
# Find one PNG
sample_img_path = None

for action_name in os.listdir(DATA_ROOT):
    action_path = os.path.join(DATA_ROOT, action_name)

    if not os.path.isdir(action_path):
        continue

    for user_name in os.listdir(action_path):
        user_path = os.path.join(action_path, user_name)

        if not os.path.isdir(user_path):
            continue

        for trial_name in os.listdir(user_path):
            trial_path = os.path.join(user_path, trial_name)

            if not os.path.isdir(trial_path):
                continue

            files = [
                f for f in os.listdir(trial_path)
                if f.lower().endswith(".png")
            ]

            if files:
                sample_img_path = os.path.join(trial_path, files[0])
                break

        if sample_img_path:
            break

    if sample_img_path:
        break

print(sample_img_path)

/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color/29_Do_squats/user17/4-1-3/Depth_2025-06-09_17-27-38.316_00000214_Color.png


In [76]:
img_bgr = cv2.imread(sample_img_path, cv2.IMREAD_COLOR)

assert img_bgr is not None

img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

depth, mask = decode_jet_depth(img_rgb)

print("Image:", img_rgb.shape, img_rgb.dtype)
print("Depth:", depth.shape, depth.dtype)
print("Mask :", mask.shape, mask.dtype)

print("Depth range:", depth.min(), depth.max())
print("Valid pixels:", mask.mean())

Image: (480, 640, 3) uint8
Depth: (480, 640) float32
Mask : (480, 640) float32
Depth range: 0.0 0.99607843
Valid pixels: 0.6969694


In [77]:
# Check that every valid pixel corresponds to a JET color

valid = mask > 0

print("Total pixels :", valid.size)
print("Valid pixels :", valid.sum())
print("Invalid pixels:", (~valid).sum())

# Verify no unknown non-black colors exist
unknown = (mask == 0) & np.any(img_rgb != 0, axis=2)

print("Unknown non-black pixels:", unknown.sum())

Total pixels : 307200
Valid pixels : 214109
Invalid pixels: 93091
Unknown non-black pixels: 0


In [78]:

def build_clip_index(users):
    clips = []

    for action_folder in sorted(os.listdir(DATA_ROOT)):
        action_path = os.path.join(DATA_ROOT, action_folder)

        if not os.path.isdir(action_path):
            continue

        # action_folder looks like: 0_Wash_face
        action_id = int(action_folder.split("_", 1)[0])

        for user in users:
            user_path = os.path.join(action_path, f"user{user}")

            if not os.path.isdir(user_path):
                continue

            for trial in sorted(os.listdir(user_path)):
                trial_path = os.path.join(user_path, trial)

                if not os.path.isdir(trial_path):
                    continue

                frames = sorted([
                    os.path.join(trial_path, f)
                    for f in os.listdir(trial_path)
                    if f.lower().endswith(".png")
                ])

                if len(frames) == 0:
                    continue

                clips.append({
                    "frames": frames,
                    "label": action_id,
                    "user": user,
                    "trial": trial
                })

    return clips


train_clips = build_clip_index(TRAIN_USERS)
val_clips = build_clip_index(VAL_USERS)

print("Train clips:", len(train_clips))
print("Val clips  :", len(val_clips))

print("\nExample:")
print(train_clips[0])

Train clips: 2490
Val clips  : 441

Example:
{'frames': ['/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color/0_Wash_face/user4/1-1-1/Depth_2025-05-08_11-43-23.116_00000092_Color.png', '/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color/0_Wash_face/user4/1-1-1/Depth_2025-05-08_11-43-23.216_00000093_Color.png', '/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color/0_Wash_face/user4/1-1-1/Depth_2025-05-08_11-43-23.316_00000094_Color.png', '/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color/0_Wash_face/user4/1-1-1/Depth_2025-05-08_11-43-23.416_00000095_Color.png', '/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color/0_Wash_face/user4/1-1-1/Depth_2025-05-08_11-43-23.516_00000096_Color.png', '/kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color/0_Wash_face/user

In [79]:
def sample_frames(frame_paths, num_frames=32, training=True):
    n = len(frame_paths)

    # --------------------------------------------------------
    # Very short clip
    # --------------------------------------------------------
    if n == 1:
        indices = np.zeros(num_frames, dtype=np.int64)

    elif n < num_frames:
        # Take all frames, then repeat to reach num_frames
        indices = np.linspace(
            0, n - 1, num_frames
        ).round().astype(np.int64)

    # --------------------------------------------------------
    # Enough frames
    # --------------------------------------------------------
    elif training:
        # Random temporal window
        start = np.random.randint(0, n - num_frames + 1)
        indices = np.arange(start, start + num_frames)

    else:
        # Deterministic uniform sampling
        indices = np.linspace(
            0, n - 1, num_frames
        ).round().astype(np.int64)

    return [frame_paths[i] for i in indices]

In [80]:
def resize_and_crop(img, size=160, training=True):
    h, w = img.shape[:2]

    scale = size / min(h, w)

    new_h = int(round(h * scale))
    new_w = int(round(w * scale))

    img = cv2.resize(
        img,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    # --------------------------------------------------------
    # Crop
    # --------------------------------------------------------
    if training:
        top = np.random.randint(0, new_h - size + 1)
        left = np.random.randint(0, new_w - size + 1)
    else:
        top = (new_h - size) // 2
        left = (new_w - size) // 2

    img = img[
        top:top + size,
        left:left + size
    ]

    return img

In [81]:
class DepthColorMotionDataset(Dataset):

    def __init__(
        self,
        clips,
        num_frames=32,
        image_size=160,
        training=True
    ):
        self.clips = clips
        self.num_frames = num_frames
        self.image_size = image_size
        self.training = training

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):

        clip = self.clips[idx]

        # ----------------------------------------------------
        # Temporal sampling
        # ----------------------------------------------------
        frame_paths = sample_frames(
            clip["frames"],
            self.num_frames,
            self.training
        )

        # ----------------------------------------------------
        # SAME spatial transform for entire clip
        # ----------------------------------------------------
        h, w = 480, 640

        scale = self.image_size / min(h, w)

        new_h = int(round(h * scale))
        new_w = int(round(w * scale))

        if self.training:
            top = np.random.randint(
                0,
                new_h - self.image_size + 1
            )
            left = np.random.randint(
                0,
                new_w - self.image_size + 1
            )
        else:
            top = (new_h - self.image_size) // 2
            left = (new_w - self.image_size) // 2

        rgb_frames = []
        depth_frames = []
        mask_frames = []

        # ----------------------------------------------------
        # Read frames
        # ----------------------------------------------------
        for path in frame_paths:

            img_bgr = cv2.imread(
                path,
                cv2.IMREAD_COLOR
            )

            if img_bgr is None:
                raise RuntimeError(
                    f"Could not read: {path}"
                )

            img_rgb = cv2.cvtColor(
                img_bgr,
                cv2.COLOR_BGR2RGB
            )

            # Exact JET decoding
            depth, mask = decode_jet_depth(img_rgb)

            # ------------------------------------------------
            # Resize
            # ------------------------------------------------
            img_rgb = cv2.resize(
                img_rgb,
                (new_w, new_h),
                interpolation=cv2.INTER_AREA
            )

            depth = cv2.resize(
                depth,
                (new_w, new_h),
                interpolation=cv2.INTER_NEAREST
            )

            mask = cv2.resize(
                mask,
                (new_w, new_h),
                interpolation=cv2.INTER_NEAREST
            )

            # ------------------------------------------------
            # SAME crop for all frames
            # ------------------------------------------------
            img_rgb = img_rgb[
                top:top+self.image_size,
                left:left+self.image_size
            ]

            depth = depth[
                top:top+self.image_size,
                left:left+self.image_size
            ]

            mask = mask[
                top:top+self.image_size,
                left:left+self.image_size
            ]

            rgb_frames.append(img_rgb)
            depth_frames.append(depth)
            mask_frames.append(mask)

        # ----------------------------------------------------
        # Stack
        # ----------------------------------------------------
        rgb = np.stack(rgb_frames)
        depth = np.stack(depth_frames)
        mask = np.stack(mask_frames)

        # ----------------------------------------------------
        # Temporal depth motion
        #
        # ΔD[t] = D[t] - D[t-1]
        #
        # First frame has zero motion.
        # ----------------------------------------------------
        motion = np.zeros_like(depth)

        motion[1:] = (
            depth[1:] - depth[:-1]
        )

        # ----------------------------------------------------
        # Normalize RGB
        # ----------------------------------------------------
        rgb = rgb.astype(np.float32) / 255.0

        # Depth already [0, ~1]
        depth = depth.astype(np.float32)

        mask = mask.astype(np.float32)

        # Motion is approximately [-1, 1].
        # Keep signed information.
        motion = motion.astype(np.float32)

        # ----------------------------------------------------
        # Convert tensors
        # ----------------------------------------------------

        # RGB
        # T,H,W,C -> C,T,H,W
        rgb = torch.from_numpy(
            rgb.transpose(3, 0, 1, 2)
        )

        # T,H,W -> T,1,H,W
        depth = torch.from_numpy(
            depth[:, None, :, :]
        )

        mask = torch.from_numpy(
            mask[:, None, :, :]
        )

        motion = torch.from_numpy(
            motion[:, None, :, :]
        )

        label = torch.tensor(
            clip["label"],
            dtype=torch.long
        )

        return {
            "rgb": rgb,
            "depth": depth,
            "mask": mask,
            "motion": motion,
            "label": label
        }

In [82]:
train_dataset_e1 = DepthColorMotionDataset(
    train_clips,
    num_frames=32,
    image_size=160,
    training=True
)

val_dataset_e1 = DepthColorMotionDataset(
    val_clips,
    num_frames=32,
    image_size=160,
    training=False
)


In [83]:
sample = train_dataset_e1[0]

print("RGB   :", sample["rgb"].shape)
print("Depth :", sample["depth"].shape)
print("Mask  :", sample["mask"].shape)
print("Motion:", sample["motion"].shape)
print("Label :", sample["label"])

print("\nRanges:")
print("RGB   :", sample["rgb"].min().item(),
      sample["rgb"].max().item())

print("Depth :", sample["depth"].min().item(),
      sample["depth"].max().item())

print("Mask  :", sample["mask"].min().item(),
      sample["mask"].max().item())

print("Motion:", sample["motion"].min().item(),
      sample["motion"].max().item())

RGB   : torch.Size([3, 32, 160, 160])
Depth : torch.Size([32, 1, 160, 160])
Mask  : torch.Size([32, 1, 160, 160])
Motion: torch.Size([32, 1, 160, 160])
Label : tensor(0)

Ranges:
RGB   : 0.0 1.0
Depth : 0.0 0.9960784316062927
Mask  : 0.0 1.0
Motion: -0.9960784316062927 0.9960784316062927


In [84]:
train_loader_e1 = DataLoader(
    train_dataset_e1,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader_e1 = DataLoader(
    val_dataset_e1,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


In [85]:
batch = next(iter(train_loader_e1))

print("RGB  :", batch["rgb"].shape)
print("Depth:", batch["depth"].shape)
print("Mask :", batch["mask"].shape)
print("Label:", batch["label"].shape)

print("\nLabels:")
print(batch["label"])

# Move to GPU
rgb = batch["rgb"].to(DEVICE, non_blocking=True)
depth = batch["depth"].to(DEVICE, non_blocking=True)
mask = batch["mask"].to(DEVICE, non_blocking=True)
labels = batch["label"].to(DEVICE, non_blocking=True)

print("\nGPU tensors:")
print("RGB device:", rgb.device)
print("Depth device:", depth.device)
print("Mask device:", mask.device)
print("Labels device:", labels.device)

RGB  : torch.Size([16, 3, 32, 160, 160])
Depth: torch.Size([16, 32, 1, 160, 160])
Mask : torch.Size([16, 32, 1, 160, 160])
Label: torch.Size([16])

Labels:
tensor([34, 23,  7, 39, 16, 20, 22,  6, 23, 26, 34, 12, 36, 20, 36, 21])

GPU tensors:
RGB device: cuda:0
Depth device: cuda:0
Mask device: cuda:0
Labels device: cuda:0


# Spatial CNN


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SpatialBackbone(nn.Module):

    def __init__(self, in_channels=6, feature_dim=192):
        super().__init__()

        self.features = nn.Sequential(

            # 160 -> 80
            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # 80 -> 40
            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # 40 -> 20
            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            # 20 -> 10
            nn.Conv2d(
                128,
                feature_dim,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(feature_dim),
            nn.ReLU(inplace=True),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):

        x = self.features(x)
        x = self.pool(x)

        return x.flatten(1)

# Temporal Residual Block


In [ ]:
class TemporalBlock(nn.Module):

    def __init__(
        self,
        channels,
        dilation,
        dropout=0.15
    ):
        super().__init__()

        padding = dilation

        self.conv1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=padding,
            dilation=dilation,
            bias=False
        )

        self.bn1 = nn.BatchNorm1d(channels)

        self.conv2 = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=padding,
            dilation=dilation,
            bias=False
        )

        self.bn2 = nn.BatchNorm1d(channels)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        residual = x

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.gelu(x)

        x = self.dropout(x)

        x = self.conv2(x)
        x = self.bn2(x)

        x = self.dropout(x)

        x = x + residual

        return F.gelu(x)

# Temporal Attention


In [ ]:
class TemporalAttention(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.score = nn.Sequential(
            nn.Linear(channels, channels // 2),
            nn.Tanh(),
            nn.Linear(channels // 2, 1)
        )

    def forward(self, x):

        # x:
        # [B, T, C]

        scores = self.score(x)

        # [B, T, 1]
        weights = torch.softmax(
            scores,
            dim=1
        )

        # Weighted temporal pooling
        pooled = torch.sum(
            x * weights,
            dim=1
        )

        return pooled, weights

# E1
## RGB + Depth + Mask + Motion
## CNN + Multi-scale TCN + Attention

In [ ]:
class E1_DepthMotion_TCN(nn.Module):

    def __init__(
        self,
        num_classes=40,
        feature_dim=192
    ):
        super().__init__()

        # ----------------------------------------------------
        # Spatial feature extraction
        # ----------------------------------------------------
        self.spatial = SpatialBackbone(
            in_channels=6,
            feature_dim=feature_dim
        )

        # ----------------------------------------------------
        # Multi-scale temporal modeling
        # ----------------------------------------------------
        self.temporal = nn.Sequential(

            # local temporal structure
            TemporalBlock(
                feature_dim,
                dilation=1
            ),

            # wider temporal structure
            TemporalBlock(
                feature_dim,
                dilation=2
            ),

            # wider again
            TemporalBlock(
                feature_dim,
                dilation=4
            ),

            # long-range structure
            TemporalBlock(
                feature_dim,
                dilation=8
            )
        )

        # ----------------------------------------------------
        # Attention
        # ----------------------------------------------------
        self.attention = TemporalAttention(
            feature_dim
        )

        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------
        self.classifier = nn.Sequential(

            nn.Linear(
                feature_dim,
                128
            ),

            nn.GELU(),

            nn.Dropout(0.30),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(
        self,
        rgb,
        depth,
        mask,
        motion
    ):

        # ----------------------------------------------------
        # RGB
        #
        # [B, 3, T, H, W]
        # ----------------------------------------------------
        B, _, T, H, W = rgb.shape

        rgb = rgb.permute(
            0, 2, 1, 3, 4
        )

        # [B,T,3,H,W]
        depth = depth

        mask = mask

        motion = motion

        # ----------------------------------------------------
        # [B,T,1,H,W]
        # ----------------------------------------------------
        depth = depth
        mask = mask
        motion = motion

        # ----------------------------------------------------
        # Concatenate modalities
        # ----------------------------------------------------
        x = torch.cat(
            [
                rgb,
                depth,
                mask,
                motion
            ],
            dim=2
        )

        # [B,T,6,H,W]
        # -> [B*T,6,H,W]

        x = x.reshape(
            B * T,
            6,
            H,
            W
        )

        # ----------------------------------------------------
        # Spatial CNN
        # ----------------------------------------------------
        x = self.spatial(x)

        # [B*T,C]
        # -> [B,T,C]

        x = x.reshape(
            B,
            T,
            -1
        )

        # ----------------------------------------------------
        # TCN expects [B,C,T]
        # ----------------------------------------------------
        x = x.transpose(
            1,
            2
        )

        x = self.temporal(x)

        # [B,C,T]
        # -> [B,T,C]

        x = x.transpose(
            1,
            2
        )

        # ----------------------------------------------------
        # Temporal attention
        # ----------------------------------------------------
        x, attention = self.attention(x)

        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------
        logits = self.classifier(x)

        return logits, attention

In [87]:
model_e1 = E1_DepthMotion_TCN(
    num_classes=40,
    feature_dim=192
).to(DEVICE)

print(model_e1)

E1_DepthMotion_TCN(
  (spatial): SpatialBackbone(
    (features): Sequential(
      (0): Conv2d(6, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (8): ReLU(inplace=True)
      (9): Conv2d(128, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (10): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (11): ReLU(inplace=True)
    )
    (pool): AdaptiveAvgPool2d(output_size=1)
  )
  (temporal): Sequent

In [88]:
num_params = sum(
    p.numel()
    for p in model_e1.parameters()
)

model_size_mb = (
    num_params * 4 / (1024 ** 2)
)

print(f"Parameters : {num_params:,}")
print(f"FP32 size  : {model_size_mb:.2f} MB")

Parameters : 1,252,201
FP32 size  : 4.78 MB


In [89]:
batch_e1 = next(iter(train_loader_e1))

rgb = batch_e1["rgb"].to(
    DEVICE,
    non_blocking=True
)

depth = batch_e1["depth"].to(
    DEVICE,
    non_blocking=True
)

mask = batch_e1["mask"].to(
    DEVICE,
    non_blocking=True
)

motion = batch_e1["motion"].to(
    DEVICE,
    non_blocking=True
)

labels = batch_e1["label"].to(
    DEVICE,
    non_blocking=True
)

print("RGB   :", rgb.shape)
print("Depth :", depth.shape)
print("Mask  :", mask.shape)
print("Motion:", motion.shape)
print("Label :", labels.shape)

RGB   : torch.Size([16, 3, 32, 160, 160])
Depth : torch.Size([16, 32, 1, 160, 160])
Mask  : torch.Size([16, 32, 1, 160, 160])
Motion: torch.Size([16, 32, 1, 160, 160])
Label : torch.Size([16])


In [90]:
model_e1.eval()

with torch.no_grad():

    logits, attention = model_e1(
        rgb,
        depth,
        mask,
        motion
    )

print("RGB     :", rgb.shape)
print("Depth   :", depth.shape)
print("Mask    :", mask.shape)
print("Motion  :", motion.shape)
print("Logits  :", logits.shape)
print("Attention:", attention.shape)

print(
    "Finite:",
    torch.isfinite(logits).all().item()
)

RGB     : torch.Size([16, 3, 32, 160, 160])
Depth   : torch.Size([16, 32, 1, 160, 160])
Mask    : torch.Size([16, 32, 1, 160, 160])
Motion  : torch.Size([16, 32, 1, 160, 160])
Logits  : torch.Size([16, 40])
Attention: torch.Size([16, 32, 1])
Finite: True


# E1 TRAINING CONFIG


In [91]:

E1_EPOCHS = 20
E1_LR = 3e-4
E1_WEIGHT_DECAY = 1e-4
E1_GRAD_CLIP = 1.0
E1_WARMUP_EPOCHS = 3

criterion_e1 = nn.CrossEntropyLoss()

optimizer_e1 = torch.optim.AdamW(
    model_e1.parameters(),
    lr=E1_LR,
    weight_decay=E1_WEIGHT_DECAY
)

scheduler_e1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_e1,
    T_max=E1_EPOCHS - E1_WARMUP_EPOCHS
)

scaler_e1 = torch.amp.GradScaler("cuda")

print("E1 configuration ready")
print("Epochs:", E1_EPOCHS)
print("LR:", E1_LR)
print("Weight decay:", E1_WEIGHT_DECAY)

E1 configuration ready
Epochs: 20
LR: 0.0003
Weight decay: 0.0001


In [92]:
num_params_e1 = sum(
    p.numel()
    for p in model_e1.parameters()
)

model_size_e1_mb = (
    num_params_e1 * 4 / (1024 ** 2)
)

print(f"E1 Parameters : {num_params_e1:,}")
print(f"E1 FP32 size  : {model_size_e1_mb:.2f} MB")

E1 Parameters : 1,252,201
E1 FP32 size  : 4.78 MB


In [93]:
import wandb

wandb.init(
    project="CIUX",
    name="E1_rgb_depth_mask_motion_tcn_attention",
    config={
        "experiment": "E1",

        "modality": "RGB + Exact JET Depth + Mask + Motion",

        "model": "CNN + MultiScale TCN + Temporal Attention",

        "num_classes": 40,
        "num_frames": 32,
        "image_size": 160,
        "batch_size": 16,

        "epochs": E1_EPOCHS,
        "learning_rate": E1_LR,
        "weight_decay": E1_WEIGHT_DECAY,
        "grad_clip": E1_GRAD_CLIP,
        "warmup_epochs": E1_WARMUP_EPOCHS,

        "feature_dim": 192,

        "train_users": TRAIN_USERS,
        "val_users": VAL_USERS,

        "e0_best_val_acc": 0.2154,
    }
)

print("W&B E1 initialized")

best_val_accuracy,▁██
epoch,▁▅█
learning_rate,▁▄█
train/accuracy,▁▅█
train/loss,█▄▁
val/accuracy,▁█▁
val/loss,▆▁█
best_val_accuracy,0.17234
epoch,3
learning_rate,0.0003
train/accuracy,0.1739


W&B E1 initialized


In [94]:
def train_one_epoch_e1(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    grad_clip=1.0
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:

        rgb = batch["rgb"].to(
            device,
            non_blocking=True
        )

        depth = batch["depth"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        motion = batch["motion"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits, _ = model(
                rgb,
                depth,
                mask,
                motion
            )

            loss = criterion(
                logits,
                labels
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            grad_clip
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        running_loss / total,
        correct / total
    )

In [95]:
@torch.no_grad()
def validate_e1(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_predictions = []
    all_labels = []

    for batch in loader:

        rgb = batch["rgb"].to(
            device,
            non_blocking=True
        )

        depth = batch["depth"].to(
            device,
            non_blocking=True
        )

        mask = batch["mask"].to(
            device,
            non_blocking=True
        )

        motion = batch["motion"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits, _ = model(
                rgb,
                depth,
                mask,
                motion
            )

            loss = criterion(
                logits,
                labels
            )

        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        all_predictions.append(
            predictions.cpu()
        )

        all_labels.append(
            labels.cpu()
        )

    all_predictions = torch.cat(
        all_predictions
    ).numpy()

    all_labels = torch.cat(
        all_labels
    ).numpy()

    return (
        running_loss / total,
        correct / total,
        all_predictions,
        all_labels
    )

# Training Loop

In [ ]:
best_val_acc_e1 = 0.0
best_epoch_e1 = 0

history_e1 = []

for epoch in range(1, E1_EPOCHS + 1):

    if epoch <= E1_WARMUP_EPOCHS:

        warmup_lr = (
            E1_LR
            * epoch
            / E1_WARMUP_EPOCHS
        )

        for param_group in optimizer_e1.param_groups:
            param_group["lr"] = warmup_lr
    train_loss, train_acc = train_one_epoch_e1(
        model_e1,
        train_loader_e1,
        criterion_e1,
        optimizer_e1,
        scaler_e1,
        DEVICE,
        E1_GRAD_CLIP
    )

    # Validation
    # --------------------------------------------------------
    val_loss, val_acc, preds, targets = validate_e1(
        model_e1,
        val_loader_e1,
        criterion_e1,
        DEVICE
    )

    # Scheduler
    # --------------------------------------------------------
    if epoch > E1_WARMUP_EPOCHS:
        scheduler_e1.step()

    current_lr = optimizer_e1.param_groups[0]["lr"]

    # Save history
    # --------------------------------------------------------
    history_e1.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": current_lr
    })

    # Best checkpoint
    # --------------------------------------------------------
    if val_acc > best_val_acc_e1:

        best_val_acc_e1 = val_acc
        best_epoch_e1 = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model_e1.state_dict(),
                "optimizer_state_dict": optimizer_e1.state_dict(),
                "val_acc": val_acc,
                "config": wandb.config.as_dict()
            },
            "/kaggle/working/E1_best.pth"
        )

        is_best = True

    else:
        is_best = False

    # --------------------------------------------------------
    # W&B
    # --------------------------------------------------------
    wandb.log({
        "epoch": epoch,

        "train/loss": train_loss,
        "train/accuracy": train_acc,

        "val/loss": val_loss,
        "val/accuracy": val_acc,

        "learning_rate": current_lr,

        "best_val_accuracy": best_val_acc_e1
    })

    print(
        f"Epoch {epoch:02d}/{E1_EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_acc:.4f} | "
        f"Val Loss {val_loss:.4f} | "
        f"Val Acc {val_acc:.4f} | "
        f"LR {current_lr:.2e}"
        + ("  ★ BEST" if is_best else "")
    )

print("\n" + "=" * 65)
print("E1 COMPLETE")
print("=" * 65)
print(f"Best validation accuracy: {best_val_acc_e1:.4f}")
print(f"Best epoch: {best_epoch_e1}")

Epoch 01/20 | Train Loss 3.3681 | Train Acc 0.1129 | Val Loss 3.1875 | Val Acc 0.1633 | LR 1.00e-04  ★ BEST
Epoch 02/20 | Train Loss 3.0934 | Train Acc 0.1486 | Val Loss 3.1359 | Val Acc 0.1655 | LR 2.00e-04  ★ BEST
Epoch 03/20 | Train Loss 2.9307 | Train Acc 0.1751 | Val Loss 4.1542 | Val Acc 0.1134 | LR 3.00e-04
Epoch 04/20 | Train Loss 2.7506 | Train Acc 0.2020 | Val Loss 2.9633 | Val Acc 0.2154 | LR 2.97e-04  ★ BEST
Epoch 05/20 | Train Loss 2.6671 | Train Acc 0.2145 | Val Loss 3.0174 | Val Acc 0.1973 | LR 2.90e-04
Epoch 06/20 | Train Loss 2.5781 | Train Acc 0.2289 | Val Loss 2.8972 | Val Acc 0.1950 | LR 2.78e-04
Epoch 07/20 | Train Loss 2.4835 | Train Acc 0.2474 | Val Loss 3.0294 | Val Acc 0.1633 | LR 2.61e-04
Epoch 08/20 | Train Loss 2.3743 | Train Acc 0.2723 | Val Loss 2.8227 | Val Acc 0.2404 | LR 2.40e-04  ★ BEST
Epoch 09/20 | Train Loss 2.3032 | Train Acc 0.2920 | Val Loss 2.8012 | Val Acc 0.2517 | LR 2.17e-04  ★ BEST
Epoch 10/20 | Train Loss 2.2673 | Train Acc 0.2888 | Val Los